In [2]:
import os

from langchain_core.messages import HumanMessage
from langchain.tools import BaseTool, StructuredTool, tool
from langchain import hub

from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [4]:
n_gpu_layers = 1  # The number of layers to put on the GPU. The rest will be on the CPU. If you don't know how many layers there are, you can use -1 to move all to GPU.
n_batch = 512  # Should be between 1 and n_ctx, consider the amount of RAM of your Apple Silicon Chip.
# Make sure the model path is correct for your system!
llm = ChatOllama(
    model="llama3.2",
    n_batch=n_batch,
    f16_kv=True,  # MUST set to True, otherwise you will run into problem after a couple of calls
    verbose=True,  # Verbose is required to pass to the callback manager
)

In [32]:
@tool
def offer(item: str, price: float):
  """
  Offer an item for sale at a given price.
  """
  print(f"OFFERING {item} FOR {price}")
  pass

@tool
def sell(item: str, price: float):
  """
  Sell an item at a given price. This can only be done after the shopkeeper has accepted an offer
  """
  print(f"SELLING {item} FOR {price}")
  pass

@tool
def rescind_offer(item: str):
  """
  Rescind an offer for an item. This means the seller is no longer willing to sell the item.
  """
  print(f"RESCINDING OFFER FOR {item}")
  pass

@tool
def leave_shop():
  """
  Leave the shop. This means the seller is no longer interested in selling anything and ends the conversation.
  """
  print("LEAVING SHOP")
  pass

In [57]:
# Initialize model
tools = [offer, sell, rescind_offer, leave_shop]
prompt_template = ChatPromptTemplate([
    ("system", """You are Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.
Inside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to drive up its value.
Erik prefers to haggle based on the uniqueness or rarity of each item, especially when he senses a merchant might undervalue magical or historical goods. He’s patient but firm in his negotiations, and while he’s willing to compromise on the mana crystal, he’s prepared to walk away if he doesn’t get a good offer for the amulet or the dagger.

You are here to haggle with the shopkeeper and try to sell your items.
"""),
    MessagesPlaceholder("msgs")
])

model_with_tools = llm.bind_tools(tools)
query = "Hello Erik! I see you have some interesting items for sale. What can I do for you today?"
messages = [HumanMessage(query)]
prompt = prompt_template.invoke({"msgs": messages})

ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T02:49:33.189711Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'offer', 'arguments': {'item': 'silver dagger etched with mysterious runes', 'price': 500}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1685059833, 'load_duration': 32357500, 'prompt_eval_count': 607, 'prompt_eval_duration': 1216230000, 'eval_count': 22, 'eval_duration': 434374000}, id='run-9be74086-d11c-467a-85c6-0de448978708-0', tool_calls=[{'name': 'offer', 'args': {'item': 'silver dagger etched with mysterious runes', 'price': 500}, 'id': '6d014dd5-02ce-4272-9ba5-bc44e17572c6', 'type': 'tool_call'}], usage_metadata={'input_tokens': 607, 'output_tokens': 22, 'total_tokens': 629})]

In [62]:
# Invoke the ai model again
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
messages

[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T02:49:33.189711Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'offer', 'arguments': {'item': 'silver dagger etched with mysterious runes', 'price': 500}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1685059833, 'load_duration': 32357500, 'prompt_eval_count': 607, 'prompt_eval_duration': 1216230000, 'eval_count': 22, 'eval_duration': 434374000}, id='run-9be74086-d11c-467a-85c6-0de448978708-0', tool_calls=[{'name': 'offer', 'args': {'item': 'silver dagger etched with mysterious runes', 'price': 500}, 'id': '6d014dd5-02ce-4272-9ba5-bc44e17572c6', 'type': 'tool_call'}], usage_metadata={'input_tokens': 607, 'output_tokens': 22, 'total_tokens': 629}),
 ToolMessage(content

In [58]:
# Call tools if the ai message contains tool calls
for tool_call in ai_msg.tool_calls:
    selected_tool = {
      "offer": offer,
      "sell": sell,
      "rescind_offer": rescind_offer,
      "leave_shop": leave_shop
    }[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

OFFERING silver dagger etched with mysterious runes FOR 500.0


In [60]:
# User query, then get a response from AI
query = "What is your name?"
messages.append(HumanMessage(query))
prompt = prompt_template.invoke({"msgs": messages})
ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T02:49:33.189711Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'offer', 'arguments': {'item': 'silver dagger etched with mysterious runes', 'price': 500}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1685059833, 'load_duration': 32357500, 'prompt_eval_count': 607, 'prompt_eval_duration': 1216230000, 'eval_count': 22, 'eval_duration': 434374000}, id='run-9be74086-d11c-467a-85c6-0de448978708-0', tool_calls=[{'name': 'offer', 'args': {'item': 'silver dagger etched with mysterious runes', 'price': 500}, 'id': '6d014dd5-02ce-4272-9ba5-bc44e17572c6', 'type': 'tool_call'}], usage_metadata={'input_tokens': 607, 'output_tokens': 22, 'total_tokens': 629}),
 ToolMessage(content